# exp145_learned_likelihood_rawtest_feature_generator_parity inference

Raw-test learned likelihood feature generator. This notebook intentionally does not create `submission.csv`.

## Contents

1. Setup and configuration
2. Raw-test replay checks
3. Raw-test feature generation
4. Schema parity readout

## 1. Setup and configuration

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import pandas as pd

from settings import EXPERIMENT_NAME, ExperimentPaths, get_nested, load_config
from learned_likelihood_rawtest_feature_generator_parity import run_generator

DEBUG = os.environ.get('EXPERIMENT_DEBUG', '0') == '1'
RAWTEST_CACHE = os.environ.get('EXPERIMENT_RAWTEST_CACHE')

paths = ExperimentPaths()
paths.require_kaggle_runtime()
paths.ensure_output_dirs()
config = load_config()

print('Experiment:', EXPERIMENT_NAME)
print('Route:', get_nested(config, 'experiment.route'))
print('Inference mode:', get_nested(config, 'inference.mode'))
print('Raw data:', get_nested(config, 'data.raw_dir'))
print('Artifacts:', paths.artifacts_dir)
print('Debug:', DEBUG, 'Provided rawtest cache:', RAWTEST_CACHE)


## 2. Raw-test replay checks

In [ ]:
raw_dir = Path(get_nested(config, 'data.raw_dir'))
test_dir = Path(get_nested(config, 'data.test_dir'))
print('raw_dir exists:', raw_dir.exists(), raw_dir)
print('test_dir exists:', test_dir.exists(), test_dir)
print('sample_submission exists:', Path(get_nested(config, 'data.sample_submission')).exists())
print('Replay settings:', get_nested(config, 'generator.rawtest_replay'))


## 3. Raw-test feature generation

In [ ]:
summary = run_generator(
    output_dir=paths.artifacts_dir,
    mode='rawtest',
    train_cache_path=None,
    rawtest_cache_path=Path(RAWTEST_CACHE) if RAWTEST_CACHE else None,
    exp111_schema_path=get_nested(config, 'data.exp111_feature_schema'),
    exp111_manifest_path=get_nested(config, 'data.exp111_model_manifest'),
    exp112_schema_path=get_nested(config, 'data.exp112_feature_schema'),
    max_rows=None,
)

print(json.dumps(summary['outputs'].get('rawtest_ml_features', {}), indent=2))


## 4. Schema parity readout

In [ ]:
parity_path = Path(summary['generated_schema']['parity_path'])
parity = pd.read_csv(parity_path)
print('schema parity pass:', summary['generated_schema']['schema_parity_pass'])
print('mismatch rows:', summary['generated_schema']['mismatch_rows'])
display(parity.head(20))

if not summary['generated_schema']['schema_parity_pass']:
    display(parity[~parity['matches_position']].head(50))

print('No submission.csv is generated by this experiment.')
